1. Read the document
2. Split the document
    - May fail to generate a response due to token limit
    - Longer documents (longer input) take more time to generate responses
3. Embedding: Store in a vector database
4. When a question is asked, perform similarity search in the vector database
5. Pass the retrieved documents along with the question to the LLM

In [ ]:
# %pip install --upgrade -q docx2txt langchain-community

In [ ]:
# %pip install -qU langchain-text-splitters

In [ ]:
# 1,2 : read&load document -> split document

from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader("./tax.docx")
# split document
document_list = loader.load_and_split(text_splitter=text_splitter)

In [ ]:
len(document_list)

In [ ]:
# pull text embedding model (text to vector)
# !ollama pull nomic-embed-text

In [ ]:
# 3: Embedding
from langchain_ollama import OllamaEmbeddings

embedding = OllamaEmbeddings(model="nomic-embed-text")

# dimension 확인용 test
# test = embedding.embed_query("test")
# print(len(test))  # output을 pinecone index의 dimensions에 넣어

In [ ]:
# %pip install --upgrade --quiet \
#     langchain-pinecone \
#     langchain-ollama \
#     langchain

In [ ]:
from dotenv import load_dotenv
import os
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore

# pinecone 연결
load_dotenv()
pinecone_api_key = os.environ.get("PINECONE_API_KEY")
pc = Pinecone(api_key=pinecone_api_key)

# pinecone index에 document_list를 embedding
database = PineconeVectorStore.from_documents(
    document_list, embedding, index_name="tax-index"
)

In [ ]:
# 4: Similarity search. Send a query (string) and retrieve the top k most similar documents (default = 4)
query = "연봉 5천만원인 직장인의 소득세는 얼마인가요?"
retrieved_docs = database.similarity_search(query=query, k=3)

In [ ]:
# 5: Pass retrieved documents along with the question to the LLM
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3:latest")

In [ ]:
# RetrievalQA Chain
# %pip install -U langchain langchainhub -q

In [ ]:
from langchain_classic import hub

# get prompt from hub
prompt = hub.pull("rlm/rag-prompt")

In [ ]:
# Build QA Chain
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm, retriever=database.as_retriever(), chain_type_kwargs={"prompt": prompt}
)

In [ ]:
ai_message = qa_chain.invoke({"query": query})

In [ ]:
ai_message

# RAG (Retrieval-Augmented Generation) Pipeline

```
User Query
    │
    ▼
Embedding Model (nomic-embed-text)
    │  Convert query to vector
    ▼
Vector DB (Pinecone)
    │  Compare query vector with stored document vectors
    │  → Return top k most similar documents
    ▼
Prompt Assembly
    │  Hub prompt + retrieved documents + user query
    │  "Answer the question based on the context below"
    │  context: [retrieved documents]
    │  question: [user query]
    ▼
LLM (llama3)
    │  Generate answer based on the prompt
    ▼
Final Answer
```

The key point is that the query **does not go directly to the LLM**. It first retrieves relevant documents from the Vector DB, then passes them along with the query to the LLM. The LLM alone doesn't know Korean tax law, but with the retrieved documents as context, it can answer. **This is the core idea of RAG.**